# GLEE Competition agent V21 — controlled larger experiments

V21 runs the tested non-LLM V19/V20 heuristic policy in repeated, interruptible
micro-batches. It targets 20 authoritative completions in each family, but enters
only one queue at a time, leaves after observing an assignment, drains all assigned
games, and checks rating, game count, fallbacks, telemetry, and assignment overshoot
before re-entering matchmaking.

The purpose is to collect a larger role- and configuration-stratified sample without
returning to V11's blocking high-concurrency runner. A family-level rating stop-loss
is enforced from the initial snapshot. Overshoot, action fallback, or telemetry
failure aborts the complete session. No Qwen or other language model is used.


In [ ]:
# Competition SDK dependency.
%pip install -q -U glee-sdk


## Configuration, memory, and append-only evidence


In [ ]:
import csv
import hashlib
import json
import math
import os
import re
import statistics
import threading
import time
from collections import defaultdict, deque
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

# Do not embed an API key in the notebook. The live cell prompts only when needed.
LOCK = threading.RLock()
SEEN = set()
DECISION_LOG = deque(maxlen=2000)
ACTION_FALLBACK_LOG = deque(maxlen=300)
TELEMETRY_LOG = deque(maxlen=500)
OUTCOME_LOG = deque(maxlen=1000)
TERMINAL_LOG = deque(maxlen=1000)

POLICY_ASSIGNMENTS = {}
POLICY_STATS = defaultdict(lambda: {"n": 0, "sum": 0.0, "sum_sq": 0.0})
REWARDED_GAMES = set()
RATING_CREDITED_GAMES = set()

# Champion mode freezes the three live-tested cores. Calibration is deliberately opt-in.
EXPLORATION_RATE = 0.06
MIN_GLOBAL_PROMOTION = 12
MIN_LOCAL_PROMOTION = 3
SAFETY_TOLERANCE = 0.012
RATING_DELTA_SCALE = 4.0
RUN_MODE = os.environ.get("GLEE_V21_MODE", "champion").strip().lower()
if RUN_MODE not in {"champion", "calibration"}:
    raise ValueError("GLEE_V21_MODE must be champion or calibration")

WORK_DIR = Path(os.environ.get("GLEE_V21_WORK_DIR", str(Path.cwd())))
WORK_DIR.mkdir(parents=True, exist_ok=True)
POLICY_STATE_PATH = Path(os.environ.get(
    "GLEE_V21_STATE_PATH", str(WORK_DIR / "glee_v21_policy_state.json")
))
EVIDENCE_PATH = Path(os.environ.get(
    "GLEE_V21_EVIDENCE_PATH", str(WORK_DIR / "glee_v21_evidence.jsonl")
))

BARGAINING_MEMORY = defaultdict(lambda: {
    "rejected_floor": 0.0, "opponent_demands": deque(maxlen=40)
})
NEGOTIATION_MEMORY = defaultdict(lambda: {
    "seller_prices": deque(maxlen=50), "buyer_prices": deque(maxlen=50)
})
PERSUASION_MEMORY = defaultdict(lambda: {
    "pos_high": 0.0, "pos_low": 0.0,
    "neg_high": 0.0, "neg_low": 0.0,
    "positive_buys": 0.0, "positive_decisions": 0.0,
    "negative_buys": 0.0, "negative_decisions": 0.0,
    "all_pos_high": 0.0, "all_pos_low": 0.0,
    "all_neg_high": 0.0, "all_neg_low": 0.0,
})

def append_jsonl(event, payload):
    """Append one crash-tolerant evidence record; serialization never breaks play."""
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "event": event,
        "payload": payload,
    }
    try:
        with LOCK:
            with EVIDENCE_PATH.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    except Exception as exc:
        TELEMETRY_LOG.append({"event": "jsonl_write", "error": f"{type(exc).__name__}: {exc}"})


def configure_glee_api_key():
    """Load the API key at runtime without storing it in notebook source."""
    if os.environ.get("GLEE_API_KEY"):
        return os.environ["GLEE_API_KEY"]
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret("GLEE_API_KEY")
    except Exception:
        secret = getpass("GLEE API key: ")
    if not secret:
        raise RuntimeError("GLEE_API_KEY was not provided")
    os.environ["GLEE_API_KEY"] = secret
    return os.environ["GLEE_API_KEY"]


## Shared schema, persistent evidence, and champion gate


In [ ]:
def clamp(x, low, high):
    return max(low, min(high, x))

def finite_float(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default

def round_progress(state):
    current = max(1, int(state.get("round", 1)))
    maximum = state.get("max_rounds")
    if state.get("horizon_known") and maximum:
        return clamp((current - 1) / max(1, int(maximum) - 1), 0.0, 1.0)
    return min(0.65, (current - 1) / 16.0)

def final_round(state):
    return bool(state.get("horizon_known") and state.get("max_rounds") and
                int(state.get("round", 1)) >= int(state["max_rounds"]))

def player_index(player):
    return 1 if player in {"player_1", "alice", "Alice"} else 2

def canonical_player(player):
    return f"player_{player_index(player)}"

def other_player(player):
    return "player_2" if player_index(player) == 1 else "player_1"

def opponent_key(game, family):
    opponent = game.get("opponent") or {}
    if opponent.get("type") != "hidden" and opponent.get("name"):
        return f"{family}:named:{opponent.get('type')}:{opponent['name']}"
    return f"{family}:game:{game.get('game_id', '')}"

def stable_unit(game, salt=""):
    state = game.get("game_state") or {}
    token = f"{game.get('game_id', '')}:{state.get('round', 1)}:{salt}"
    value = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return value / 2**64

def logistic(x):
    return 1.0 / (1.0 + math.exp(-clamp(x, -60.0, 60.0)))

def allocation(offer, player):
    keys = (("player_1_gain", "alice_gain") if player_index(player) == 1 else
            ("player_2_gain", "bob_gain"))
    for key in keys:
        if key in offer:
            return finite_float(offer[key], None)
    return None

def action_message(text):
    return str(text)[:2000]

def signal_polarity(value):
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower().replace("_", " ")
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]+", " ", text)).strip()
    negatives = (
        "do not buy", "don't buy", "should not buy", "shouldn't buy",
        "do not recommend", "don't recommend", "recommend against",
        "not worth", "not a good", "not high quality", "low quality",
        "bad product", "buying would be foolish",
    )
    positives = (
        "buy this", "recommend buying", "recommend this", "worth it",
        "high quality", "great product", "good product", "positive",
    )
    if text in {"no", "false", "negative", "not recommended"}:
        return False
    words = set(text.split())
    if any(phrase in text for phrase in negatives) or words & {"pass", "skip", "avoid"}:
        return False
    if text in {"yes", "true", "recommended"}:
        return True
    if text == "buy" or any(phrase in text for phrase in positives):
        return True
    return None

def tail_repeat_count(values, relative_tolerance=0.002):
    if not values:
        return 0
    last = float(values[-1])
    tolerance = max(1e-9, abs(last) * relative_tolerance)
    count = 0
    for value in reversed(values):
        if abs(float(value) - last) <= tolerance:
            count += 1
        else:
            break
    return count

def coarse_bucket(value, edges):
    value = finite_float(value)
    return sum(value >= edge for edge in edges)

def policy_context(game, role):
    state = game["game_state"]
    family = game["game_family"]
    info = "full" if state.get("complete_information") else "hidden"
    horizon = int(state.get("max_rounds", state.get("total_rounds", 0)) or 0)
    horizon_bucket = coarse_bucket(horizon, (5, 12, 30))
    if family == "bargaining":
        own_delta = state.get(f"delta_{player_index(role)}", 0.93)
        return (role, info, horizon_bucket,
                coarse_bucket(own_delta, (0.80, 0.95, 0.99)))
    if family == "negotiation":
        return (role, info, horizon_bucket)
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    price = finite_float(state.get("product_price"), 0.0)
    v = finite_float(state.get("v"), max(price, 1.0))
    u = finite_float(state.get("u"), 0.0)
    cutoff = (price - u) / (v - u) if v > u else 1.0
    mode = state.get("seller_message_type") or game.get("valid_actions", {}).get("type", "unknown")
    known_values = "known" if "v" in state and "u" in state else "hidden"
    return (role, str(mode), known_values, horizon_bucket,
            coarse_bucket(p, (0.33, 0.66)), coarse_bucket(cutoff, (0.35, 0.65)))

def protected_arm(family, role):
    if family == "bargaining":
        return "v4_safe"
    if family == "negotiation":
        return "v5_safe"
    return "v3_safe"

def candidate_arms(family, role):
    if family == "bargaining" and player_index(role) == 1:
        return ("alice_claim_006", "alice_deal_006")
    if family == "persuasion" and role == "seller":
        return ("seller_truthful", "seller_terminal_pool")
    return ()

def stat_key(scope, family, role, context, arm):
    payload = [scope, family, role, list(context) if context is not None else None, arm]
    return json.dumps(payload, separators=(",", ":"), sort_keys=False)

def summary_from_item(item):
    n = int(item.get("n", 0))
    mean = item.get("sum", 0.0) / n if n else 0.5
    variance = max(0.0, item.get("sum_sq", 0.0) / n - mean * mean) if n else 0.25
    radius = 1.64 * math.sqrt((variance + 0.02) / max(1, n))
    return n, mean, clamp(mean - radius, 0.0, 1.0), clamp(mean + radius, 0.0, 1.0)

def arm_summary(family, role, context, arm, scope="local"):
    key = stat_key(scope, family, role, context if scope == "local" else None, arm)
    return summary_from_item(POLICY_STATS[key])

def candidate_is_eligible(family, role, context, candidate, baseline):
    local_c = arm_summary(family, role, context, candidate, "local")
    global_c = arm_summary(family, role, context, candidate, "global")
    local_b = arm_summary(family, role, context, baseline, "local")
    global_b = arm_summary(family, role, context, baseline, "global")
    promoted = (global_c[0] >= MIN_GLOBAL_PROMOTION and
                local_c[0] >= MIN_LOCAL_PROMOTION and
                global_b[0] >= MIN_GLOBAL_PROMOTION and
                local_b[0] >= MIN_LOCAL_PROMOTION and
                global_c[2] >= global_b[2] - SAFETY_TOLERANCE and
                local_c[2] >= local_b[2] - SAFETY_TOLERANCE and
                global_c[1] > global_b[1] + 0.01)
    clearly_worse = (global_c[0] >= 6 and global_b[0] >= 6 and
                     global_c[3] < global_b[2])
    return promoted, not clearly_worse

def is_synthetic_game_id(game_id):
    return str(game_id).startswith((
        "test-", "v5-", "v6-", "v7-", "v8-", "v13-",
        "bad-", "synthetic-"
    ))

def select_policy_arm(game, role):
    game_id = str(game.get("game_id", "unknown"))
    with LOCK:
        if game_id in POLICY_ASSIGNMENTS:
            return POLICY_ASSIGNMENTS[game_id]["arm"]
        family = game["game_family"]
        context = policy_context(game, role)
        baseline = protected_arm(family, role)
        arm = baseline
        candidates = candidate_arms(family, role)
        synthetic = is_synthetic_game_id(game_id)
        if RUN_MODE == "calibration" and candidates and not synthetic:
            promoted = []
            exploratory = []
            for candidate in candidates:
                is_promoted, may_explore = candidate_is_eligible(
                    family, role, context, candidate, baseline
                )
                if is_promoted:
                    promoted.append(candidate)
                elif may_explore:
                    exploratory.append(candidate)
            if promoted:
                arm = max(promoted, key=lambda name: arm_summary(
                    family, role, context, name, "global")[1])
            elif exploratory and stable_unit(game, f"v13-explore:{role}") < EXPLORATION_RATE:
                index = min(len(exploratory) - 1,
                            int(stable_unit(game, "v13-candidate") * len(exploratory)))
                arm = exploratory[index]
        POLICY_ASSIGNMENTS[game_id] = {
            "family": family, "role": role, "context": context, "arm": arm,
            "player": canonical_player(game.get(
                "your_player", game["game_state"].get("current_player", "player_1"))),
            "state": dict(game["game_state"]), "synthetic": synthetic,
        }
        return arm

def policy_state_payload():
    return {"version": 13, "objective": "dashboard_rating_delta", "stats": {key: dict(value) for key, value in POLICY_STATS.items()}}

def load_policy_state(path=POLICY_STATE_PATH):
    if not Path(path).exists():
        return 0
    try:
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        if payload.get("version") != 13 or not isinstance(payload.get("stats"), dict):
            raise ValueError("incompatible policy-state version")
        for key, value in payload["stats"].items():
            if isinstance(value, dict) and int(value.get("n", 0)) >= 0:
                POLICY_STATS[key] = {
                    "n": int(value.get("n", 0)),
                    "sum": finite_float(value.get("sum")),
                    "sum_sq": max(0.0, finite_float(value.get("sum_sq"))),
                }
        return len(payload["stats"])
    except Exception as exc:
        TELEMETRY_LOG.append({"stage": "state_load", "error": repr(exc)})
        return 0

def save_policy_state(path=POLICY_STATE_PATH):
    try:
        path = Path(path)
        temporary = path.with_suffix(path.suffix + ".tmp")
        temporary.write_text(json.dumps(policy_state_payload(), indent=2), encoding="utf-8")
        os.replace(temporary, path)
        return True
    except Exception as exc:
        TELEMETRY_LOG.append({"stage": "state_save", "error": repr(exc)})
        return False

def _find_numeric(mapping, names):
    if not isinstance(mapping, dict):
        return None
    for name in names:
        value = mapping.get(name)
        if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value):
            return float(value)
    for value in mapping.values():
        if isinstance(value, dict):
            found = _find_numeric(value, names)
            if found is not None:
                return found
    return None

def normalized_terminal_reward(assignment, payload):
    player = assignment["player"]
    role = assignment["role"]
    idx = player_index(player)
    payoff = _find_numeric(payload, (
        f"{player}_payoff", f"player_{idx}_payoff", f"payoff_{idx}",
        f"{role}_total_payoff", "your_payoff", "payoff", "utility", "score",
    ))
    if payoff is None:
        return None
    state = assignment["state"]
    family = assignment["family"]
    if family == "bargaining":
        scale = max(1.0, abs(finite_float(state.get("money_to_divide"), 1.0)))
        return clamp(payoff / scale, 0.0, 1.0)
    if family == "negotiation":
        values = [abs(finite_float(v)) for k, v in state.items() if k.endswith("_value")]
        return clamp(payoff / max([1.0] + values), 0.0, 1.0)
    rounds = max(1, int(state.get("total_rounds", 1) or 1))
    if role == "seller":
        scale = max(1.0, rounds * abs(finite_float(state.get("product_price"), 1.0)))
        return clamp(payoff / scale, 0.0, 1.0)
    scale = max(1.0, rounds * max(abs(finite_float(state.get("v"), 1.0)),
                                  abs(finite_float(state.get("u"), 0.0)),
                                  abs(finite_float(state.get("product_price"), 1.0))))
    return clamp(0.5 + 0.5 * math.tanh(payoff / scale), 0.0, 1.0)

def update_policy_reward(assignment, reward):
    family, role, context, arm = (assignment["family"], assignment["role"],
                                  assignment["context"], assignment["arm"])
    for scope in ("local", "global"):
        key = stat_key(scope, family, role, context if scope == "local" else None, arm)
        item = POLICY_STATS[key]
        item["n"] += 1
        item["sum"] += reward
        item["sum_sq"] += reward * reward

def harvest_completed_games(client):
    harvested = []
    for game_id, assignment in list(POLICY_ASSIGNMENTS.items()):
        if game_id in REWARDED_GAMES or assignment.get("synthetic"):
            continue
        try:
            payload = client.game_state(game_id)
            reward = normalized_terminal_reward(assignment, payload)
            if reward is None:
                TELEMETRY_LOG.append({"stage": "reward_missing", "game_id": game_id})
                continue
            with LOCK:
                REWARDED_GAMES.add(game_id)
                record = {"game_id": game_id, "family": assignment["family"],
                          "role": assignment["role"], "arm": assignment["arm"],
                          "context": assignment["context"],
                          "terminal_reward": round(reward, 6)}
                TERMINAL_LOG.append(record)
            harvested.append(record)
        except Exception as exc:
            TELEMETRY_LOG.append({"stage": "reward_harvest", "game_id": game_id,
                                  "error": repr(exc)})
    return harvested

def rating_delta_reward(delta, scale=RATING_DELTA_SCALE):
    # 0.5 means no rating change; the transform bounds outliers without
    # discarding their sign. This is used only for conservative arm comparison.
    return logistic(finite_float(delta) / max(1e-9, finite_float(scale, 4.0)))

def credit_rating_delta(game_ids, family, before_rating, after_rating, count_delta,
                        persist=True):
    eligible = []
    for game_id in game_ids:
        assignment = POLICY_ASSIGNMENTS.get(str(game_id))
        if (assignment and not assignment.get("synthetic") and
                assignment.get("family") == family and
                str(game_id) not in RATING_CREDITED_GAMES):
            eligible.append((str(game_id), assignment))
    if count_delta != 1 or len(eligible) != 1:
        return None
    game_id, assignment = eligible[0]
    delta = finite_float(after_rating) - finite_float(before_rating)
    reward = rating_delta_reward(delta)
    with LOCK:
        update_policy_reward(assignment, reward)
        RATING_CREDITED_GAMES.add(game_id)
        record = {"game_id": game_id, "family": family,
                  "role": assignment["role"], "arm": assignment["arm"],
                  "context": assignment["context"],
                  "rating_delta": round(delta, 6),
                  "rating_reward": round(reward, 6)}
        OUTCOME_LOG.append(record)
    if persist:
        save_policy_state()
    return record

def policy_evidence_rows():
    rows = []
    for key, item in POLICY_STATS.items():
        scope, family, role, context, arm = json.loads(key)
        n, mean, lower, upper = summary_from_item(item)
        rows.append({"scope": scope, "family": family, "role": role,
                     "context": context, "arm": arm, "n": n,
                     "mean": round(mean, 5), "lower": round(lower, 5),
                     "upper": round(upper, 5)})
    return sorted(rows, key=lambda row: (row["family"], row["role"], row["scope"], row["arm"]))

def checkpoint_stop_reason(count_delta, new_action_fallbacks, new_telemetry,
                           current_rating, initial_rating, stop_loss,
                           abort_on_overshoot=True):
    if count_delta == 0:
        return "no observed completion"
    if abort_on_overshoot and count_delta > 1:
        return f"completion overshoot: {count_delta}"
    if new_action_fallbacks:
        return "live action fallback"
    if new_telemetry:
        return "new telemetry diagnostic"
    if current_rating < initial_rating - stop_loss:
        return "rating stop-loss"
    return None

def export_session_report(json_path="glee_v13_session_report.json",
                          csv_path="glee_v13_arm_outcomes.csv"):
    live_assignments = {
        game_id: data for game_id, data in POLICY_ASSIGNMENTS.items()
        if not data.get("synthetic")
    }
    payload = {
        "version": 13, "run_mode": RUN_MODE,
        "assignments": live_assignments,
        "rating_outcomes": list(OUTCOME_LOG),
        "terminal_outcomes": list(TERMINAL_LOG),
        "policy_evidence": policy_evidence_rows(),
        "action_fallbacks": list(ACTION_FALLBACK_LOG),
        "telemetry": list(TELEMETRY_LOG),
    }
    json_path = Path(json_path)
    temporary = json_path.with_suffix(json_path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    os.replace(temporary, json_path)

    csv_path = Path(csv_path)
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=(
            "game_id", "family", "role", "arm", "rating_delta", "rating_reward", "context"
        ))
        writer.writeheader()
        for item in OUTCOME_LOG:
            writer.writerow({
                "game_id": item["game_id"], "family": item["family"],
                "role": item["role"], "arm": item["arm"],
                "rating_delta": item["rating_delta"],
                "rating_reward": item["rating_reward"],
                "context": json.dumps(item["context"], separators=(",", ":")),
            })
    return json_path.resolve(), csv_path.resolve()

LOADED_POLICY_ROWS = load_policy_state()
print("Loaded persistent V13 policy rows:", LOADED_POLICY_ROWS)


## 1. Bargaining — V4 champion with cycle insurance


In [ ]:
def player_delta(state, player, default=0.93):
    return clamp(finite_float(state.get(f"delta_{player_index(player)}", default), default),
                 0.001, 0.9999)

def rubinstein_responder_share(proposer_delta, responder_delta):
    denominator = 1.0 - proposer_delta * responder_delta
    proposer_share = ((1.0 - responder_delta) / denominator
                      if denominator > 1e-12 else 0.5)
    return clamp(1.0 - proposer_share, 0.001, 0.999)

def bargaining_offer_series(state, money):
    series = {"player_1": [], "player_2": []}
    if money <= 0:
        return series
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        offer = record.get("offer") or {}
        proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
        demand = allocation(offer, proposer)
        if demand is not None:
            series[proposer].append(clamp(demand / money, 0.0, 1.0))
    return series

def bargaining_stall_count(state, money):
    series = bargaining_offer_series(state, money)
    return min(tail_repeat_count(series["player_1"], 0.003),
               tail_repeat_count(series["player_2"], 0.003))

def bargaining_profile_key(game, state, me):
    # Reservation shares are configuration dependent. Never carry a hard
    # rejection floor across roles or incompatible discount conditions.
    info = "complete" if state.get("complete_information") else "hidden"
    own_delta = round(player_delta(state, me), 3)
    opponent = other_player(me)
    visible_opponent_delta = state.get(f"delta_{player_index(opponent)}")
    opponent_delta = (round(finite_float(visible_opponent_delta), 3)
                      if visible_opponent_delta is not None else "x")
    family = (f"bargaining:{canonical_player(me)}:{info}:"
              f"d{own_delta}:od{opponent_delta}")
    return opponent_key(game, family)

def update_bargaining_memory(game, me, opponent, money):
    key = bargaining_profile_key(game, game["game_state"], me)
    with LOCK:
        model = BARGAINING_MEMORY[key]
        for record in game["game_state"].get("history", []):
            if not isinstance(record, dict):
                continue
            offer = record.get("offer") or {}
            proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
            decision = record.get("decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("bargaining-v13", game.get("game_id"), record.get("round"),
                     proposer, str(decision),
                     tuple(sorted((str(k), str(v)) for k, v in offer.items())))
            if event in SEEN:
                continue
            SEEN.add(event)
            if proposer == canonical_player(me) and str(decision).lower() == "reject":
                rejected = allocation(offer, opponent)
                if rejected is not None and money > 0:
                    model["rejected_floor"] = max(model["rejected_floor"], rejected / money)
            if proposer == canonical_player(opponent):
                demand = allocation(offer, opponent)
                if demand is not None and money > 0:
                    model["opponent_demands"].append(clamp(demand / money, 0.0, 1.0))
        return {"rejected_floor": model["rejected_floor"],
                "opponent_demands": list(model["opponent_demands"])}

def estimate_bargaining_floor(game, state, me, opponent, model):
    t = round_progress(state)
    if state.get("complete_information"):
        prior = rubinstein_responder_share(player_delta(state, me),
                                           player_delta(state, opponent))
    else:
        opponent_type = (game.get("opponent") or {}).get("type")
        prior = (0.46 if opponent_type == "human" else 0.43) + 0.02 * t
    evidence = model["rejected_floor"] + 0.006 if model["rejected_floor"] else 0.0
    if model["opponent_demands"]:
        recent = model["opponent_demands"][-5:]
        demand_floor = statistics.median(recent) - (0.065 - 0.020 * t)
        evidence = max(evidence, demand_floor)
    return clamp(max(prior, evidence), 0.20, 0.999)

def bargaining_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    money = finite_float(state["money_to_divide"])
    t = round_progress(state)
    my_delta = player_delta(state, me)
    model = update_bargaining_memory(game, me, opponent, money)
    floor = estimate_bargaining_floor(game, state, me, opponent, model)
    stalls = bargaining_stall_count(state, money)
    arm = select_policy_arm(game, me)

    if game["valid_actions"]["type"] == "offer":
        if stalls >= 3 and model["opponent_demands"]:
            # Match the opponent's revealed demand rather than repeating a split
            # they have already rejected indefinitely.
            responder_share = clamp(model["opponent_demands"][-1], 0.20, 0.9999)
        else:
            best = None
            alice = player_index(me) == 1
            if arm == "alice_claim_006" and alice:
                shade, payoff_power, failure_multiplier = 0.006, 1.17, 0.90
            elif arm == "alice_deal_006" and alice:
                shade, payoff_power, failure_multiplier = -0.006, 1.13, 1.10
            else:  # exact V4 policy constants; always used by Bob
                shade, payoff_power, failure_multiplier = 0.0, 1.15, 1.0
            search_floor = clamp(floor - shade, 0.18, 0.999)
            failure_cost = ((0.04 + 0.24 * t + 0.55 * (1.0 - my_delta)) *
                            failure_multiplier)
            # 10% through 99.5% in half-percentage-point increments.
            for step in range(20, 200):
                responder_share = step / 200.0
                width = 0.012 if search_floor > 0.80 else 0.022
                probability = logistic((responder_share - search_floor + 0.008) / width)
                own_share = 1.0 - responder_share
                objective = probability * own_share**payoff_power - (1.0 - probability) * failure_cost
                candidate = (objective, own_share, responder_share)
                if best is None or candidate > best:
                    best = candidate
            responder_share = best[2]
        responder_gain = round(money * responder_share, 8)
        own_gain = money - responder_gain
        action = ({"alice_gain": own_gain, "bob_gain": responder_gain}
                  if player_index(me) == 1 else
                  {"alice_gain": responder_gain, "bob_gain": own_gain})
        if state.get("messages_allowed"):
            pct = round(100 * responder_share, 1)
            action["message"] = action_message(
                f"I offer you {pct}% now, accounting for discounting and the observed negotiation path."
            )
        return action

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        return {"decision": "reject"}
    if final_round(state):
        return {"decision": "accept" if current_gain >= 0 else "reject"}
    if stalls >= 3 and current_gain > 0:
        return {"decision": "accept"}
    next_own_share = 1.0 - floor
    deal_probability = clamp(0.80 + 0.10 * t - 0.20 * model["rejected_floor"], 0.45, 0.92)
    continuation_share = my_delta * next_own_share * deal_probability
    candidate_margin = {"alice_claim_006": 0.006,
                        "alice_deal_006": -0.006}.get(arm, 0.0)
    role_margin = (candidate_margin * (1.0 - t)
                   if player_index(me) == 1 and stalls == 0 else 0.0)
    risk_floor = max(0.0, 0.34 + role_margin - 0.08 * t - 0.07 * stalls)
    required = money * max(risk_floor, continuation_share)
    return {"decision": "accept" if current_gain + 1e-9 >= required else "reject"}


## 2. Negotiation — exact V5/V8 role-calibrated champion


In [ ]:
def offer_sender(item, record, field):
    sender = item.get("from_player") if isinstance(item, dict) else None
    if sender:
        return canonical_player(sender)
    if field == "counteroffer" and record.get("decided_by"):
        return canonical_player(record["decided_by"])
    return None

def negotiation_price_series(state):
    series = {"player_1": [], "player_2": []}
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        for field in ("offer", "counteroffer"):
            item = record.get(field)
            if isinstance(item, dict) and item.get("price") is not None:
                sender = offer_sender(item, record, field)
                if sender:
                    series[sender].append(finite_float(item["price"]))
    return series

def negotiation_stall_count(state, me, opponent):
    series = negotiation_price_series(state)
    return min(tail_repeat_count(series[canonical_player(me)], 0.001),
               tail_repeat_count(series[canonical_player(opponent)], 0.001))

def negotiation_profile_key(game, state):
    me = canonical_player(game.get("your_player", state["current_player"]))
    role = state.get(f"{me}_role", "unknown")
    info = "complete" if state.get("complete_information") else "hidden"
    return opponent_key(game, f"negotiation:{role}:{info}")

def update_negotiation_memory(game, state):
    key = negotiation_profile_key(game, state)
    with LOCK:
        model = NEGOTIATION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            for field in ("offer", "counteroffer"):
                item = record.get(field)
                if not isinstance(item, dict) or item.get("price") is None:
                    continue
                sender = offer_sender(item, record, field)
                if sender is None:
                    continue
                price = finite_float(item["price"])
                event = ("negotiation-v13", game.get("game_id"), record.get("round"),
                         field, sender, price)
                if event in SEEN:
                    continue
                SEEN.add(event)
                role = state.get(f"{sender}_role")
                if role in {"seller", "buyer"}:
                    model[f"{role}_prices"].append(price)
        return {name: list(values) for name, values in model.items()}

def opponent_prices_in_game(state, opponent):
    return negotiation_price_series(state)[canonical_player(opponent)]

def projected_opponent_price(prices, role):
    if not prices:
        return None
    latest = prices[-1]
    if len(prices) < 2:
        return latest
    step = latest - prices[-2]
    # Only extrapolate concessions in the economically expected direction.
    if role == "seller":
        step = min(0.0, step)
    else:
        step = max(0.0, step)
    return max(0.0, latest + 0.6 * step)

def negotiation_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    role = state[f"{me}_role"]
    select_policy_arm(game, role)  # records the protected whole-game arm
    opponent_role = state[f"{opponent}_role"]
    my_value = finite_float(state[f"{me}_value"])
    t = round_progress(state)
    update_negotiation_memory(game, state)
    observed = opponent_prices_in_game(state, opponent)
    stalls = negotiation_stall_count(state, me, opponent)
    opponent_value = state.get(f"{opponent}_value")
    surplus = None

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = finite_float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        own_capture = ((0.74 - 0.14 * t) if role == "seller" else
                       (0.66 - 0.10 * t))
        if observed and surplus > 1e-12:
            last_price = observed[-1]
            opponent_demand = ((last_price - seller_value) / surplus if opponent_role == "seller"
                               else (buyer_value - last_price) / surplus)
            feasible_capture = 1.0 - clamp(opponent_demand - (0.05 + 0.04 * t), 0.0, 1.0)
            own_capture = 0.58 * own_capture + 0.42 * feasible_capture
        own_capture = clamp(own_capture, 0.52, 0.82)
        if surplus <= 0:
            target = my_value
        elif role == "seller":
            target = seller_value + own_capture * surplus
        else:
            target = buyer_value - own_capture * surplus
    elif observed:
        anchor = projected_opponent_price(observed, opponent_role)
        claim = ((0.70 - 0.12 * t) if role == "seller" else
                 (0.62 - 0.10 * t))
        if role == "seller" and anchor >= my_value:
            target = my_value + claim * (anchor - my_value)
        elif role == "buyer" and anchor <= my_value:
            target = my_value - claim * (my_value - anchor)
        else:
            target = my_value
    elif role == "seller":
        target = my_value * (1.36 - 0.16 * t)
    else:
        target = my_value * (0.80 + 0.10 * t)

    target = max(0.0, finite_float(target, my_value))
    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This price is inside the feasible interval revealed by our offers."
            )
        return action

    price = finite_float((state.get("last_offer") or {}).get("price"), my_value)
    offered_utility = price - my_value if role == "seller" else my_value - price
    profitable = offered_utility >= -1e-9
    if final_round(state):
        return {"decision": "AcceptOffer" if profitable else "RejectOffer"}
    if profitable and stalls >= 2:
        return {"decision": "AcceptOffer"}
    if not profitable and stalls >= 3 and not state.get("horizon_known"):
        return {"decision": "WalkAway"}

    target_utility = abs(target - my_value)
    offered_capture = offered_utility / surplus if surplus is not None and surplus > 1e-12 else None
    continuation_base = 0.80 if role == "seller" else 0.75
    continuation = target_utility * (continuation_base - 0.12 * t -
                                     0.08 * min(stalls, 2))
    if profitable and (offered_utility + 1e-9 >= continuation or
                       (offered_capture is not None and offered_capture >= 0.50 + 0.05 * (1 - t))):
        return {"decision": "AcceptOffer"}

    blend = 0.24 + 0.43 * t + 0.08 * min(stalls, 2)
    blend = clamp(blend, 0.0, 0.78)
    counter = (1.0 - blend) * target + blend * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(max(0.0, counter), 8)}
    if state.get("messages_allowed"):
        action["message"] = action_message(
            "I am conceding inside my individually rational range."
        )
    return action


## 3. Persuasion — V3 Bayesian buyer and credibility-constrained seller


In [ ]:
def persuasion_profile_key(game, seller_view):
    state = game["game_state"]
    perspective = "buyer-response" if seller_view else "seller-reliability"
    action_type = game.get("valid_actions", {}).get("type", "")
    mode = state.get("seller_message_type") or (
        "text" if action_type == "seller_message" else "binary"
    )
    return opponent_key(game, f"persuasion:v13:{perspective}:{mode}")

def update_persuasion_memory(game, seller_view):
    state = game["game_state"]
    key = persuasion_profile_key(game, seller_view)
    with LOCK:
        model = PERSUASION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            signal = signal_polarity(record.get("seller_message"))
            quality = record.get("quality")
            decision = record.get("buyer_decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            decision_text = str(decision).lower()
            bought = record.get("bought") is True or decision_text == "yes"
            event = ("persuasion-v13", "seller" if seller_view else "buyer",
                     game.get("game_id"), record.get("round"), signal,
                     quality if bought else "unobserved", decision_text)
            if event in SEEN:
                continue
            SEEN.add(event)
            # Preserve V3 seller credit exactly in all_* counts while the
            # response-gated challenger uses buyer-visible purchased counts.
            if seller_view and signal is not None and quality in {"high", "low"}:
                model[f"all_{'pos' if signal else 'neg'}_{quality}"] += 1.0
            # Buyer-visible precision changes only after purchase.
            if bought and signal is not None and quality in {"high", "low"}:
                model[f"{'pos' if signal else 'neg'}_{quality}"] += 1.0
            if seller_view and signal is not None and decision_text in {"yes", "no"}:
                prefix = "positive" if signal else "negative"
                model[f"{prefix}_decisions"] += 1.0
                model[f"{prefix}_buys"] += float(decision_text == "yes")
        return dict(model)

def strategic_signal_prior(positive, p):
    q_high, q_low = ((0.90, 0.24) if positive else (0.10, 0.76))
    denominator = p * q_high + (1.0 - p) * q_low
    return p * q_high / denominator if denominator > 1e-12 else p

def smoothed_signal_precision(model, positive, p):
    high = model["pos_high"] if positive else model["neg_high"]
    low = model["pos_low"] if positive else model["neg_low"]
    prior = strategic_signal_prior(positive, p)
    return (4.0 * prior + high) / (4.0 + high + low)

def beta_lower_mean(successes, trials, alpha=1.5, beta=0.5):
    a = alpha + successes
    b = beta + max(0.0, trials - successes)
    mean = a / (a + b)
    variance = a * b / ((a + b) ** 2 * (a + b + 1.0))
    return clamp(mean - 0.65 * math.sqrt(max(0.0, variance)), 0.0, 1.0)

def persuasion_cutoff(state, price):
    if "v" not in state or "u" not in state:
        return None
    v, u = finite_float(state["v"]), finite_float(state["u"])
    if v <= u:
        return None
    return clamp((price - u) / (v - u), 0.0, 1.0)

def static_pool_bound(p, cutoff):
    if cutoff is None or not (0.0 < p < 1.0) or not (0.0 < cutoff < 1.0):
        return 0.0
    return clamp(p * (1.0 - cutoff) / (cutoff * (1.0 - p)), 0.0, 1.0)

def v3_pool_probability(game, state, model, p, price, t, round_no, total_rounds):
    cutoff = persuasion_cutoff(state, price)
    if cutoff is None:
        return 0.0
    if cutoff <= 0.0:
        return 1.0
    if cutoff >= 1.0:
        return 0.0
    static_pool = static_pool_bound(p, cutoff)
    response_rate = ((model["positive_buys"] + 1.5) /
                     (model["positive_decisions"] + 2.0))
    ramp = 0.12 + 0.88 * t**1.65
    probability = clamp(static_pool * ramp * (0.55 + 0.60 * response_rate), 0.0, 1.0)
    precision_after_lie = ((2.5 + model["all_pos_high"]) /
                           (2.5 + 0.8 + model["all_pos_high"] +
                            model["all_pos_low"] + 1.0))
    credible = precision_after_lie >= min(0.97, cutoff + 0.025)
    prefix_done = round_no > max(1, math.ceil(0.12 * total_rounds))
    return probability if credible and prefix_done else 0.0

def terminal_pool_probability(state, model, p, price, t, round_no, total_rounds):
    # No deception before the final two periods. On the last period there is no
    # future reputation cost, but pooling still requires evidence that the buyer
    # acts on positive messages and respects the static obedience bound.
    remaining = total_rounds - round_no
    if remaining > 1:
        return 0.0
    response_lcb = beta_lower_mean(model["positive_buys"],
                                   model["positive_decisions"])
    if model["positive_decisions"] < 2 or response_lcb < 0.45:
        return 0.0
    cutoff = persuasion_cutoff(state, price)
    if cutoff is None:
        return 0.12 if remaining == 0 and response_lcb >= 0.60 else 0.0
    if cutoff <= 0.0:
        return 1.0
    if cutoff >= 1.0:
        return 0.0
    bound = static_pool_bound(p, cutoff)
    cap = 1.0 if remaining == 0 else 0.35
    return clamp(bound * response_lcb, 0.0, min(cap, bound))

def persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = finite_float(state["product_price"])
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", round_no)))
    t = (round_no - 1) / max(1, total_rounds - 1)
    seller_view = action_type in {"seller_message", "seller_recommendation"}
    role = "seller" if seller_view else "buyer"
    arm = select_policy_arm(game, role)
    model = update_persuasion_memory(game, seller_view)

    if seller_view:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low":
            if arm == "seller_truthful":
                probability = 0.0
            elif arm == "seller_terminal_pool":
                probability = terminal_pool_probability(
                    state, model, p, price, t, round_no, total_rounds
                )
            else:
                probability = v3_pool_probability(
                    game, state, model, p, price, t, round_no, total_rounds
                )
            recommend = stable_unit(game, f"v13-pool:{arm}") < probability
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        return {"message": action_message(
            "My recommendation: BUY this product." if recommend else
            "My recommendation: PASS on this product this round."
        )}

    # Exact V3 buyer decision core, with observation-correct history.
    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return {"decision": "yes"}
    if price > v:
        return {"decision": "no"}
    signal = signal_polarity(state.get("seller_message"))
    posterior = p if signal is None else smoothed_signal_precision(model, signal, p)
    expected_value = posterior * v + (1.0 - posterior) * u
    observations = (model["pos_high"] + model["pos_low"] if signal is True else
                    model["neg_high"] + model["neg_low"] if signal is False else 0.0)
    remaining_fraction = (total_rounds - round_no) / max(1, total_rounds)
    information_bonus = (0.012 * max(0.0, v - u) * remaining_fraction /
                         math.sqrt(1.0 + observations) if signal is True else 0.0)
    return {"decision": "yes" if expected_value + information_bonus >= price else "no"}


## Strict validated dispatcher and safe fallback


In [ ]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def fallback_action(game):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        if action_type == "offer":
            money = finite_float(state["money_to_divide"])
            alice = round(money / 2.0, 8)
            return {"alice_gain": alice, "bob_gain": money - alice}
        return {"decision": "accept"}
    if family == "negotiation":
        me = canonical_player(game.get("your_player", state["current_player"]))
        role = state[f"{me}_role"]
        value = finite_float(state[f"{me}_value"])
        if action_type == "offer":
            return {"product_price": max(0.0, value)}
        price = finite_float((state.get("last_offer") or {}).get("price"), value)
        profitable = price >= value if role == "seller" else price <= value
        if profitable:
            return {"decision": "AcceptOffer"}
        if final_round(state):
            return {"decision": "RejectOffer"}
        return {"decision": "RejectOffer", "product_price": max(0.0, value)}
    if action_type == "seller_message":
        return {"message": "My recommendation: PASS this round."}
    if action_type == "seller_recommendation":
        return {"decision": "no"}
    p = finite_float(state.get("p"), 0.5)
    expected = p * finite_float(state.get("v")) + (1 - p) * finite_float(state.get("u"))
    return {"decision": "yes" if expected >= finite_float(state["product_price"]) else "no"}

def is_finite_number(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool) and
            math.isfinite(float(value)))

def contract_action_keys(game, action):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        keys = {"alice_gain", "bob_gain"} if action_type == "offer" else {"decision"}
    elif family == "negotiation":
        if action_type == "offer":
            keys = {"product_price"}
        else:
            keys = {"decision"}
            if action.get("decision") == "RejectOffer" and not final_round(state):
                keys.add("product_price")
    elif action_type == "seller_message":
        keys = {"message"}
    else:
        keys = {"decision"}
    if (family in {"bargaining", "negotiation"} and
            state.get("messages_allowed")):
        keys.add("message")
    return keys

def validate_action(game, action):
    if not isinstance(action, dict):
        raise ValueError("strategy must return a dict")
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    allowed = contract_action_keys(game, action)
    if set(action) - allowed:
        raise ValueError(f"unexpected action keys: {sorted(set(action) - allowed)}")
    declared = game["valid_actions"].get("fields")
    if isinstance(declared, dict) and declared:
        undeclared = set(action) - set(declared)
        if undeclared:
            raise ValueError(f"keys absent from valid_actions.fields: {sorted(undeclared)}")
    if "message" in action and not isinstance(action["message"], str):
        raise ValueError("message must be a string")
    if family == "bargaining" and action_type == "offer":
        if not is_finite_number(action.get("alice_gain")) or not is_finite_number(action.get("bob_gain")):
            raise ValueError("bargaining gains must be finite numbers")
        alice = float(action["alice_gain"])
        bob = float(action["bob_gain"])
        pot = finite_float(state["money_to_divide"])
        if not all(math.isfinite(x) and x >= 0 for x in (alice, bob)):
            raise ValueError("invalid bargaining allocation")
        if not math.isclose(alice + bob, pot, rel_tol=1e-10, abs_tol=1e-7):
            raise ValueError("bargaining gains do not sum to the pot")
    elif family == "bargaining":
        if action.get("decision") not in {"accept", "reject", "walkaway"}:
            raise ValueError("invalid bargaining decision")
    elif family == "negotiation" and action_type == "offer":
        if (not is_finite_number(action.get("product_price")) or
                float(action["product_price"]) < 0):
            raise ValueError("invalid negotiation price")
    elif family == "negotiation":
        if action.get("decision") not in {"AcceptOffer", "RejectOffer", "WalkAway"}:
            raise ValueError("invalid negotiation decision")
        if action["decision"] == "RejectOffer" and not final_round(state):
            if (not is_finite_number(action.get("product_price")) or
                    float(action["product_price"]) < 0):
                raise ValueError("counteroffer required")
    elif action_type == "seller_message":
        if not isinstance(action.get("message"), str) or len(action["message"]) > 2000:
            raise ValueError("invalid persuasion message")
    elif action.get("decision") not in {"yes", "no"}:
        raise ValueError("invalid persuasion decision")
    if "message" in action and len(action["message"]) > 2000:
        raise ValueError("message exceeds 2,000 characters")
    return action

def strategy(game):
    error = None
    used_fallback = False
    try:
        family = game["game_family"]
        action = validate_action(game, STRATEGIES[family](game))
    except Exception as exc:
        used_fallback = True
        error = f"{type(exc).__name__}: {exc}"
        action = validate_action(game, fallback_action(game))
        ACTION_FALLBACK_LOG.append({"game_id": game.get("game_id"),
                                    "family": game.get("game_family"),
                                    "round": (game.get("game_state") or {}).get("round"),
                                    "error": error})
        print(f"SAFE ACTION FALLBACK {game.get('game_id')}: {error}")
    with LOCK:
        DECISION_LOG.append({
            "game_id": game.get("game_id"), "family": game.get("game_family"),
            "round": (game.get("game_state") or {}).get("round"),
            "action": dict(action), "used_fallback": used_fallback, "error": error,
        })
    append_jsonl("decision", {
        "game_id": game.get("game_id"), "family": game.get("game_family"),
        "round": (game.get("game_state") or {}).get("round"),
        "action": action, "used_fallback": used_fallback, "error": error,
    })
    return action


## Offline contract, champion-lock, and property tests


In [ ]:
def base_game(family, action_type, state, player="player_1", game_id="v13-test",
              opponent=None):
    return {
        "game_id": game_id, "game_family": family, "your_player": player,
        "opponent": opponent or {"type": "hidden", "name": None},
        "valid_actions": {"type": action_type, "fields": {}},
        "game_state": state,
    }

def force_arm(game, role, arm):
    POLICY_ASSIGNMENTS[str(game["game_id"])] = {
        "family": game["game_family"], "role": role,
        "context": policy_context(game, role), "arm": arm,
        "player": canonical_player(game["your_player"]),
        "state": dict(game["game_state"]), "synthetic": True,
    }

class HarvestAuditClient:
    def __init__(self):
        self.calls = []
    def game_state(self, game_id):
        self.calls.append(game_id)
        return {"player_1_payoff": 60.0}

def run_v13_tests():
    global RUN_MODE
    original_mode = RUN_MODE
    RUN_MODE = "champion"
    template = {"current_player": "player_1", "round": 1, "max_rounds": 5,
                "horizon_known": True, "money_to_divide": 100,
                "delta_1": 0.9, "delta_2": 0.95,
                "complete_information": True, "history": [],
                "messages_allowed": True}
    for player in ("player_1", "player_2"):
        state = dict(template, current_player=player)
        game = base_game("bargaining", "offer", state, player, f"v13-b-{player}")
        action = strategy(game)
        assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)
        assert POLICY_ASSIGNMENTS[game["game_id"]]["arm"] == "v4_safe"

    extreme = dict(template, max_rounds=0, horizon_known=False,
                   delta_1=0.9, delta_2=0.999)
    assert strategy(base_game("bargaining", "offer", extreme,
                              "player_1", "v13-b-extreme"))["bob_gain"] > 90

    # A live-like Alice game is champion-locked even though candidates exist.
    champion_game = base_game("bargaining", "offer", template,
                              "player_1", "champion-lock-live-like")
    champion_action = strategy(champion_game)
    assert POLICY_ASSIGNMENTS[champion_game["game_id"]]["arm"] == "v4_safe"
    assert math.isclose(champion_action["alice_gain"] + champion_action["bob_gain"], 100)
    POLICY_ASSIGNMENTS[champion_game["game_id"]]["synthetic"] = True

    # Alice challengers remain feasible and Bob has no eligible challenger.
    for arm in ("alice_claim_006", "alice_deal_006"):
        game = base_game("bargaining", "offer", template,
                         "player_1", f"synthetic-{arm}")
        force_arm(game, "player_1", arm)
        action = strategy(game)
        assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)
    assert candidate_arms("bargaining", "player_2") == ()

    # Property sweep: every pot and visible discount pair produces an exact budget.
    for pot in (1.0, 37.5, 100.0, 1_000_000.0):
        for d1, d2 in ((0.5, 0.5), (0.9, 0.999), (0.999, 0.7)):
            for player in ("player_1", "player_2"):
                state = dict(template, current_player=player, money_to_divide=pot,
                             delta_1=d1, delta_2=d2)
                action = strategy(base_game(
                    "bargaining", "offer", state, player,
                    f"v13-sweep-b-{pot}-{d1}-{d2}-{player}"))
                assert action["alice_gain"] >= 0 and action["bob_gain"] >= 0
                assert math.isclose(action["alice_gain"] + action["bob_gain"],
                                    pot, rel_tol=1e-10, abs_tol=1e-7)

    for seller_value, buyer_value in ((0, 100), (40, 100), (99, 100)):
        for player in ("player_1", "player_2"):
            role = "seller" if player == "player_1" else "buyer"
            state = {"current_player": player, "player_1_role": "seller",
                     "player_2_role": "buyer", "player_1_value": seller_value,
                     "player_2_value": buyer_value, "complete_information": True,
                     "round": 1, "max_rounds": 5, "horizon_known": True,
                     "history": [], "messages_allowed": False}
            game = base_game("negotiation", "offer", state, player,
                             f"v13-n-{seller_value}-{buyer_value}-{player}")
            action = strategy(game)
            assert seller_value <= action["product_price"] <= buyer_value
            assert POLICY_ASSIGNMENTS[game["game_id"]]["arm"] == "v5_safe"
            assert candidate_arms("negotiation", role) == ()

    # Unknown-horizon infeasible repetition exits; a zero-utility price is accepted.
    repeated = [{"round": i + 1,
                 "offer": {"price": 101, "from_player": "player_1"},
                 "decision": "RejectOffer", "decided_by": "player_2",
                 "counteroffer": {"price": 100, "from_player": "player_2"}}
                for i in range(4)]
    stalled = {"current_player": "player_2", "player_1_role": "seller",
               "player_2_role": "buyer", "player_2_value": 100,
               "complete_information": False, "round": 9,
               "horizon_known": False, "last_offer": {"price": 101},
               "history": repeated}
    assert strategy(base_game("negotiation", "decision", stalled,
                              "player_2", "v13-n-stall"))["decision"] == "WalkAway"
    zero_offer = dict(stalled, last_offer={"price": 100})
    assert strategy(base_game("negotiation", "decision", zero_offer,
                              "player_2", "v13-n-zero"))["decision"] == "AcceptOffer"

    high = {"current_quality": "high", "product_price": 50, "p": 0.5,
            "v": 100, "u": 0, "round": 1, "total_rounds": 10, "history": []}
    high_game = base_game("persuasion", "seller_recommendation", high,
                          "player_1", "v13-p-high")
    assert strategy(high_game) == {"decision": "yes"}
    assert POLICY_ASSIGNMENTS[high_game["game_id"]]["arm"] == "v3_safe"
    text_game = base_game("persuasion", "seller_message", high,
                          "player_1", "v13-p-text")
    assert "BUY" in strategy(text_game)["message"]

    low = dict(high, current_quality="low", round=5)
    truthful = base_game("persuasion", "seller_recommendation", low,
                         "player_1", "synthetic-truthful")
    force_arm(truthful, "seller", "seller_truthful")
    assert strategy(truthful) == {"decision": "no"}
    gated = base_game("persuasion", "seller_recommendation", low,
                      "player_1", "synthetic-gated")
    force_arm(gated, "seller", "seller_terminal_pool")
    assert strategy(gated)["decision"] in {"yes", "no"}

    expensive = {"seller_message": "I recommend buying this product.",
                 "product_price": 101, "p": 0.9, "v": 100, "u": 0,
                 "round": 1, "total_rounds": 5, "history": []}
    assert strategy(base_game("persuasion", "buyer_decision", expensive,
                              "player_2", "v13-p-expensive")) == {"decision": "no"}
    assert candidate_arms("persuasion", "buyer") == ()

    # A passed product does not update buyer-visible precision.
    named = {"type": "agent", "name": "v13-audit-opponent"}
    passed = {"seller_message": {"decision": "yes"}, "product_price": 50,
              "p": 0.5, "v": 100, "u": 0, "round": 2, "total_rounds": 5,
              "history": [{"round": 1, "seller_message": {"decision": "yes"},
                           "buyer_decision": "no", "bought": False, "quality": "low"}]}
    pass_game = base_game("persuasion", "buyer_decision", passed,
                          "player_2", "v13-p-pass", named)
    strategy(pass_game)
    assert PERSUASION_MEMORY[persuasion_profile_key(pass_game, False)]["pos_low"] == 0

    # Synthetic assignments are never queried, fixing V7's false fallback count.
    audit = HarvestAuditClient()
    before_calls = len(audit.calls)
    harvest_completed_games(audit)
    assert len(audit.calls) == before_calls
    assert not ACTION_FALLBACK_LOG

    assignment = POLICY_ASSIGNMENTS["synthetic-alice_claim_006"]
    assert math.isclose(normalized_terminal_reward(
        assignment, {"player_1_payoff": 60}), 0.6)
    # Promotion evidence is aligned to an unambiguous dashboard rating delta.
    rating_game = base_game("negotiation", "offer", {
        "current_player": "player_1", "player_1_role": "seller",
        "player_2_role": "buyer", "player_1_value": 40,
        "player_2_value": 100, "complete_information": True,
        "round": 1, "max_rounds": 5, "horizon_known": True,
        "history": [], "messages_allowed": False,
    }, "player_1", "rating-credit-live-like")
    strategy(rating_game)
    before_n = arm_summary("negotiation", "seller",
                           policy_context(rating_game, "seller"), "v5_safe", "global")[0]
    credited = credit_rating_delta([rating_game["game_id"]], "negotiation",
                                   1400.0, 1402.0, 1, persist=False)
    assert credited["rating_delta"] == 2.0 and credited["rating_reward"] > 0.5
    after_n = arm_summary("negotiation", "seller",
                          policy_context(rating_game, "seller"), "v5_safe", "global")[0]
    assert after_n == before_n + 1
    assert credit_rating_delta([rating_game["game_id"]], "negotiation",
                               1402.0, 1404.0, 1, persist=False) is None
    ambiguous = base_game("negotiation", "offer", rating_game["game_state"],
                          "player_1", "rating-ambiguous-live-like")
    strategy(ambiguous)
    assert credit_rating_delta([ambiguous["game_id"]], "negotiation",
                               1404.0, 1410.0, 2, persist=False) is None
    POLICY_ASSIGNMENTS[rating_game["game_id"]]["synthetic"] = True
    POLICY_ASSIGNMENTS[ambiguous["game_id"]]["synthetic"] = True

    payload = policy_state_payload()
    assert payload["version"] == 13
    assert payload["objective"] == "dashboard_rating_delta"
    assert isinstance(payload["stats"], dict)
    assert checkpoint_stop_reason(0, False, False, 100, 100, 3)
    assert checkpoint_stop_reason(2, False, False, 100, 100, 3)
    assert checkpoint_stop_reason(1, True, False, 100, 100, 3)
    assert checkpoint_stop_reason(1, False, True, 100, 100, 3)
    assert checkpoint_stop_reason(1, False, False, 96, 100, 3)
    assert checkpoint_stop_reason(1, False, False, 98, 100, 3) is None

    assert signal_polarity("Buying would be foolish; do not buy.") is False
    assert signal_polarity("My recommendation: BUY this product.") is True
    invalid = base_game("bargaining", "offer", template, "player_1", "bad-bool")
    try:
        validate_action(invalid, {"alice_gain": True, "bob_gain": 99})
        raise AssertionError("boolean gain was accepted")
    except ValueError:
        pass
    assert not ACTION_FALLBACK_LOG, list(ACTION_FALLBACK_LOG)
    RUN_MODE = original_mode
    print("All V13 champion-lock, protected-policy, telemetry, reward, and contract tests passed.")

run_v13_tests()


## V21 heuristic challenger definitions


In [ ]:
V21_ARM = "v21_heuristic"
HEURISTIC_TRACE = deque(maxlen=4000)

def robust_median_step(values, window=5):
    recent = [finite_float(value) for value in values[-window:]]
    if len(recent) < 2:
        return 0.0
    return statistics.median([right - left for left, right in zip(recent, recent[1:])])

def robust_mad(values):
    if len(values) < 2:
        return 0.0
    center = statistics.median(values)
    return statistics.median([abs(value - center) for value in values])

def assign_v21(game, role):
    game_id = str(game.get("game_id", "unknown"))
    with LOCK:
        if game_id not in POLICY_ASSIGNMENTS:
            POLICY_ASSIGNMENTS[game_id] = {
                "family": game["game_family"], "role": role,
                "context": policy_context(game, role), "arm": V21_ARM,
                "player": canonical_player(game.get(
                    "your_player", game["game_state"].get("current_player", "player_1"))),
                "state": dict(game["game_state"]),
                "synthetic": is_synthetic_game_id(game_id) or game_id.startswith("v21-"),
            }
        return V21_ARM

def trace_v21(game, role, metrics, action):
    record = {
        "game_id": str(game.get("game_id")), "family": game["game_family"],
        "role": role, "round": game["game_state"].get("round"),
        "metrics": metrics, "action": dict(action),
    }
    HEURISTIC_TRACE.append(record)
    append_jsonl("v21_heuristic_decision", record)
    return action


# ---------------------------------------------------------------------------
# Bargaining: robust trend + volatility-aware offer choice + cycle insurance
# ---------------------------------------------------------------------------
def v21_bargaining_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    assign_v21(game, me)
    money = finite_float(state["money_to_divide"])
    t = round_progress(state)
    my_delta = player_delta(state, me)
    model = update_bargaining_memory(game, me, opponent, money)
    base_floor = estimate_bargaining_floor(game, state, me, opponent, model)
    demands = model["opponent_demands"][-7:]
    trend = robust_median_step(demands)
    volatility = robust_mad(demands)
    if demands:
        predicted_demand = clamp(demands[-1] + trend, 0.0, 1.0)
        behavioral_floor = predicted_demand - (0.045 + 0.025 * t)
        floor = max(model["rejected_floor"] + (0.004 if model["rejected_floor"] else 0.0),
                    0.58 * base_floor + 0.42 * behavioral_floor)
    else:
        predicted_demand = None
        floor = base_floor
    floor = clamp(floor, 0.18, 0.999)
    stalls = bargaining_stall_count(state, money)

    if game["valid_actions"]["type"] == "offer":
        if stalls >= 3 and demands:
            responder_share = clamp(demands[-1], 0.20, 0.9999)
            reason = "cycle_match"
        else:
            alice = player_index(me) == 1
            # Alice's repeated negative split motivates slightly more deal value;
            # Bob keeps the stronger historical payoff curvature.
            payoff_power = 1.12 if alice else 1.16
            failure_multiplier = 1.12 if alice else 1.00
            search_floor = clamp(floor - (0.006 if alice else 0.0), 0.16, 0.999)
            width = clamp(0.014 + 0.85 * volatility + (0.006 if not demands else 0.0),
                          0.012, 0.055)
            failure_cost = ((0.05 + 0.28 * t + 0.58 * (1.0 - my_delta)) *
                            failure_multiplier)
            best = None
            for step in range(20, 200):
                candidate_share = step / 200.0
                accept_probability = logistic(
                    (candidate_share - search_floor + 0.006) / width
                )
                own_share = 1.0 - candidate_share
                value = (accept_probability * own_share**payoff_power -
                         (1.0 - accept_probability) * failure_cost)
                candidate = (value, own_share, candidate_share)
                if best is None or candidate > best:
                    best = candidate
            responder_share = best[2]
            reason = "expected_payoff"
        responder_gain = round(money * responder_share, 8)
        own_gain = money - responder_gain
        action = ({"alice_gain": own_gain, "bob_gain": responder_gain}
                  if player_index(me) == 1 else
                  {"alice_gain": responder_gain, "bob_gain": own_gain})
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This offer reflects the observed concession path and remaining delay cost."
            )
        return trace_v21(game, me, {
            "reason": reason, "floor": round(floor, 5), "trend": round(trend, 5),
            "volatility": round(volatility, 5), "stalls": stalls,
            "predicted_demand": predicted_demand,
        }, action)

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        action = {"decision": "reject"}
        return trace_v21(game, me, {"reason": "missing_offer"}, action)
    if final_round(state) or (stalls >= 3 and current_gain > 0):
        action = {"decision": "accept" if current_gain >= 0 else "reject"}
        return trace_v21(game, me, {"reason": "terminal_or_cycle", "stalls": stalls}, action)

    favorable_concession = -trend if demands else 0.0
    wait_bonus = clamp(0.30 * favorable_concession, -0.025, 0.025)
    continuation_share = my_delta * (1.0 - floor) * clamp(
        0.77 + wait_bonus + 0.12 * t - 0.07 * stalls, 0.42, 0.92
    )
    role_floor = 0.30 if player_index(me) == 1 else 0.33
    risk_floor = max(0.0, role_floor - 0.09 * t - 0.075 * stalls)
    required = money * max(risk_floor, continuation_share)
    decision = "accept" if current_gain + 1e-9 >= required else "reject"
    return trace_v21(game, me, {
        "reason": "continuation_test", "required": required,
        "current_gain": current_gain, "stalls": stalls,
    }, {"decision": decision})


# ---------------------------------------------------------------------------
# Negotiation: robust projection + drift-resistant surplus protection
# ---------------------------------------------------------------------------
def v21_negotiation_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    role = state[f"{me}_role"]
    opponent_role = state[f"{opponent}_role"]
    assign_v21(game, role)
    my_value = finite_float(state[f"{me}_value"])
    t = round_progress(state)
    update_negotiation_memory(game, state)
    observed = opponent_prices_in_game(state, opponent)
    stalls = negotiation_stall_count(state, me, opponent)
    raw_step = robust_median_step(observed)
    favorable_step = raw_step if role == "seller" else -raw_step
    opponent_value = state.get(f"{opponent}_value")
    surplus = None

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = finite_float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        base_capture = ((0.70 - 0.14 * t) if role == "seller" else
                        (0.63 - 0.12 * t))
        if surplus > 1e-12 and observed:
            concession_rate = clamp(favorable_step / surplus, -0.10, 0.10)
            # Hold somewhat firmer while the opponent is moving toward us;
            # concede when the path is stagnant and the horizon advances.
            base_capture += 0.35 * concession_rate
            base_capture -= 0.025 * min(stalls, 2) * (0.35 + t)
        own_capture = clamp(base_capture, 0.50, 0.76)
        if surplus <= 0:
            target = my_value
        elif role == "seller":
            target = seller_value + own_capture * surplus
        else:
            target = buyer_value - own_capture * surplus
    elif observed:
        latest = observed[-1]
        direction_step = min(0.0, raw_step) if opponent_role == "seller" else max(0.0, raw_step)
        projected = max(0.0, latest + 0.50 * direction_step)
        claim = ((0.66 - 0.13 * t) if role == "seller" else
                 (0.59 - 0.11 * t))
        if role == "seller" and projected >= my_value:
            target = my_value + claim * (projected - my_value)
        elif role == "buyer" and projected <= my_value:
            target = my_value - claim * (my_value - projected)
        else:
            target = my_value
    elif role == "seller":
        target = my_value * (1.32 - 0.14 * t)
    else:
        target = my_value * (0.82 + 0.09 * t)

    target = max(0.0, finite_float(target, my_value))
    metrics = {
        "target": target, "stalls": stalls, "opponent_step": raw_step,
        "observed_prices": observed[-5:], "surplus": surplus,
    }
    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This price remains feasible and reflects the concession path."
            )
        return trace_v21(game, role, {**metrics, "reason": "target_offer"}, action)

    price = finite_float((state.get("last_offer") or {}).get("price"), my_value)
    offered_utility = price - my_value if role == "seller" else my_value - price
    profitable = offered_utility >= -1e-9
    if final_round(state):
        action = {"decision": "AcceptOffer" if profitable else "RejectOffer"}
        return trace_v21(game, role, {**metrics, "reason": "final_round"}, action)
    if profitable and stalls >= 2:
        return trace_v21(game, role, {**metrics, "reason": "profitable_stall"},
                         {"decision": "AcceptOffer"})
    if not profitable and stalls >= 3 and not state.get("horizon_known"):
        return trace_v21(game, role, {**metrics, "reason": "infeasible_cycle"},
                         {"decision": "WalkAway"})

    target_utility = abs(target - my_value)
    continuation_factor = clamp(
        (0.76 if role == "seller" else 0.71) - 0.16 * t - 0.10 * min(stalls, 2),
        0.38, 0.80,
    )
    continuation = target_utility * continuation_factor
    capture = offered_utility / surplus if surplus is not None and surplus > 1e-12 else None
    minimum_capture = 0.47 + 0.06 * (1.0 - t)
    if profitable and (offered_utility + 1e-9 >= continuation or
                       (capture is not None and capture >= minimum_capture)):
        return trace_v21(game, role, {**metrics, "reason": "accept_threshold",
                                      "offered_utility": offered_utility},
                         {"decision": "AcceptOffer"})

    blend = clamp(0.28 + 0.44 * t + 0.10 * min(stalls, 2), 0.0, 0.82)
    counter = (1.0 - blend) * target + blend * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(max(0.0, counter), 8)}
    if state.get("messages_allowed"):
        action["message"] = action_message(
            "I can move toward your offer while preserving non-negative utility."
        )
    return trace_v21(game, role, {**metrics, "reason": "counteroffer",
                                  "offered_utility": offered_utility}, action)


# ---------------------------------------------------------------------------
# Persuasion: conservative buyer + response-gated credibility spending
# ---------------------------------------------------------------------------
def precision_lower_bound(model, positive, p, width=0.55):
    high = model["pos_high"] if positive else model["neg_high"]
    low = model["pos_low"] if positive else model["neg_low"]
    prior = strategic_signal_prior(positive, p)
    alpha = 4.0 * prior + high
    beta = 4.0 * (1.0 - prior) + low
    mean = alpha / (alpha + beta)
    variance = alpha * beta / ((alpha + beta) ** 2 * (alpha + beta + 1.0))
    return clamp(mean - width * math.sqrt(max(0.0, variance)), 0.0, 1.0)

def v21_seller_pool_probability(state, model, p, price, round_no, total_rounds):
    t = (round_no - 1) / max(1, total_rounds - 1)
    truthful_prefix = max(2, math.ceil(0.25 * total_rounds))
    if round_no <= truthful_prefix:
        return 0.0
    response_lcb = beta_lower_mean(model["positive_buys"],
                                   model["positive_decisions"], 1.0, 1.0)
    if model["positive_decisions"] < 3 or response_lcb < 0.52:
        return 0.0
    cutoff = persuasion_cutoff(state, price)
    terminal = round_no >= total_rounds
    if cutoff is None:
        return clamp((0.10 + 0.30 * t) * response_lcb, 0.0, 0.45 if terminal else 0.20)
    if cutoff <= 0.0:
        return 1.0
    if cutoff >= 1.0:
        return 0.0
    bound = static_pool_bound(p, cutoff)
    trust_after = ((2.5 + model["all_pos_high"]) /
                   (3.3 + model["all_pos_high"] + model["all_pos_low"] + 1.0))
    if not terminal and trust_after < cutoff + 0.04:
        return 0.0
    schedule = 0.10 + 0.90 * t**2
    cap = 1.0 if terminal else 0.55
    return clamp(bound * response_lcb * schedule, 0.0, min(cap, bound))

def v21_persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = finite_float(state["product_price"])
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", state.get("max_rounds", round_no)) or round_no))
    seller_view = action_type in {"seller_message", "seller_recommendation"}
    role = "seller" if seller_view else "buyer"
    assign_v21(game, role)
    model = update_persuasion_memory(game, seller_view)

    if seller_view:
        quality = state.get("current_quality", state.get("quality"))
        probability = 0.0
        recommend = quality == "high"
        if quality == "low":
            probability = v21_seller_pool_probability(
                state, model, p, price, round_no, total_rounds
            )
            recommend = stable_unit(game, "v21-response-gated-pool") < probability
        if action_type == "seller_recommendation":
            action = {"decision": "yes" if recommend else "no"}
        else:
            action = {"message": action_message(
                "My recommendation: BUY this product." if recommend else
                "My recommendation: PASS on this product this round."
            )}
        return trace_v21(game, role, {
            "quality": quality, "pool_probability": probability,
            "positive_response_trials": model["positive_decisions"],
        }, action)

    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return trace_v21(game, role, {"reason": "price_below_low_value"}, {"decision": "yes"})
    if price > v:
        return trace_v21(game, role, {"reason": "price_above_high_value"}, {"decision": "no"})
    signal = signal_polarity(state.get("seller_message"))
    posterior = p if signal is None else precision_lower_bound(model, signal, p)
    expected_value = posterior * v + (1.0 - posterior) * u
    observations = (model["pos_high"] + model["pos_low"] if signal is True else
                    model["neg_high"] + model["neg_low"] if signal is False else 0.0)
    remaining_fraction = (total_rounds - round_no) / max(1, total_rounds)
    near_cutoff = abs(expected_value - price) <= 0.08 * max(1.0, v - u)
    information_bonus = (0.010 * max(0.0, v - u) * remaining_fraction /
                         math.sqrt(1.0 + observations)
                         if signal is True and near_cutoff and observations < 4 else 0.0)
    decision = "yes" if expected_value + information_bonus >= price else "no"
    return trace_v21(game, role, {
        "signal": signal, "posterior_lcb": posterior,
        "expected_value": expected_value, "information_bonus": information_bonus,
    }, {"decision": decision})


STRATEGIES = {
    "bargaining": v21_bargaining_strategy,
    "negotiation": v21_negotiation_strategy,
    "persuasion": v21_persuasion_strategy,
}
print("Activated V21 heuristic challenger for all three families.")


## V21 cross-family property and safety tests


In [ ]:
def run_v21_tests():
    # Bargaining budget, both roles, broad discount grid.
    for player in ("player_1", "player_2"):
        for d1 in (0.05, 0.50, 0.95, 0.999):
            for d2 in (0.05, 0.50, 0.95, 0.999):
                state = {
                    "current_player": player, "money_to_divide": 137.25,
                    "delta_1": d1, "delta_2": d2, "complete_information": True,
                    "round": 1, "max_rounds": 8, "horizon_known": True,
                    "history": [], "messages_allowed": False,
                }
                game = base_game("bargaining", "offer", state, player,
                                 f"v21-b-{player}-{d1}-{d2}")
                action = strategy(game)
                assert validate_action(game, action) == action
                assert math.isclose(action["alice_gain"] + action["bob_gain"], 137.25,
                                    rel_tol=1e-10, abs_tol=1e-7)

    # The cycle detector accepts a strictly positive allocation after three stalls.
    history = []
    for turn in range(1, 4):
        history.extend([
            {"round": turn, "proposer": "player_1",
             "offer": {"player_1_gain": 35.0, "player_2_gain": 65.0}, "decision": "reject"},
            {"round": turn, "proposer": "player_2",
             "offer": {"player_1_gain": 0.01, "player_2_gain": 99.99}, "decision": "reject"},
        ])
    cycle_state = {
        "current_player": "player_1", "money_to_divide": 100.0,
        "delta_1": 0.10, "delta_2": 0.999, "complete_information": True,
        "round": 7, "horizon_known": False, "history": history,
        "last_offer": {"player_1_gain": 0.01, "player_2_gain": 99.99},
        "messages_allowed": False,
    }
    cycle_game = base_game("bargaining", "respond", cycle_state,
                           "player_1", "v21-cycle")
    assert strategy(cycle_game)["decision"] == "accept"

    # Negotiation never knowingly accepts negative utility and accepts profitable
    # final offers for both roles.
    for role, own_value, good_price, bad_price in (
        ("seller", 40.0, 45.0, 35.0), ("buyer", 100.0, 95.0, 105.0)
    ):
        other_role = "buyer" if role == "seller" else "seller"
        other_value = 100.0 if role == "seller" else 40.0
        template = {
            "current_player": "player_1", "player_1_role": role,
            "player_2_role": other_role, "player_1_value": own_value,
            "player_2_value": other_value, "complete_information": True,
            "round": 5, "max_rounds": 5, "horizon_known": True,
            "history": [], "messages_allowed": False,
        }
        good = base_game("negotiation", "respond",
                         dict(template, last_offer={"price": good_price}),
                         "player_1", f"v21-n-good-{role}")
        bad = base_game("negotiation", "respond",
                        dict(template, last_offer={"price": bad_price}),
                        "player_1", f"v21-n-bad-{role}")
        assert strategy(good)["decision"] == "AcceptOffer"
        assert strategy(bad)["decision"] == "RejectOffer"

    # Persuasion dominance and truthful-prefix behavior.
    buyer_template = {
        "current_player": "player_1", "player_1_role": "buyer",
        "player_2_role": "seller", "p": 0.5, "v": 100.0, "u": 20.0,
        "round": 1, "total_rounds": 8, "history": [],
        "seller_message": "My recommendation: BUY this product.",
    }
    cheap = base_game("persuasion", "buyer_decision",
                      dict(buyer_template, product_price=15.0),
                      "player_1", "v21-p-cheap")
    expensive = base_game("persuasion", "buyer_decision",
                          dict(buyer_template, product_price=110.0),
                          "player_1", "v21-p-expensive")
    assert strategy(cheap)["decision"] == "yes"
    assert strategy(expensive)["decision"] == "no"

    seller_state = {
        "current_player": "player_1", "player_1_role": "seller",
        "player_2_role": "buyer", "p": 0.5, "v": 100.0, "u": 0.0,
        "product_price": 50.0, "current_quality": "low",
        "round": 1, "total_rounds": 8, "history": [],
    }
    early_low = base_game("persuasion", "seller_recommendation", seller_state,
                          "player_1", "v21-p-early-low")
    assert strategy(early_low)["decision"] == "no"

    assert not ACTION_FALLBACK_LOG
    assert all(item["arm"] == V21_ARM for game_id, item in POLICY_ASSIGNMENTS.items()
               if game_id.startswith("v21-"))
    print("All V21 heuristic property, safety, and contract tests passed.")

run_v21_tests()


In [ ]:
print("V21 heuristic trace rows:", len(HEURISTIC_TRACE))
display(list(HEURISTIC_TRACE)[-30:])
print("Action fallbacks:", len(ACTION_FALLBACK_LOG))
print("Telemetry diagnostics:", len(TELEMETRY_LOG))
print("Important: offline property tests establish safety invariants, not rating superiority.")


## Controlled live runner — one queue entry, immediate leave, full drain

This driver intentionally avoids `client.run()`. It queues one family once, leaves that queue immediately when the first pending assignment is observed, and continues polling until every assigned game is finished. A pre-assignment timeout is safe; after assignment the driver drains instead of abandoning a game.

Set `RUN_LIVE=True` only after the offline tests pass. Evaluate one family at a time and start with the dashboard reporting zero active games.


In [ ]:
from glee_sdk import GleeClient

def run_one_queue_assignment(client, strategy_fn, family, poll_interval=2.0,
                             match_timeout=600.0, stats_interval=10.0):
    if family not in STRATEGIES:
        raise ValueError(f"unknown family: {family}")
    before = client.stats()
    if int(before.get("active_games", 0) or 0) != 0:
        raise RuntimeError("start only when the dashboard reports zero active games")

    assigned_ids = set()
    completed_ids = set()
    move_results = []
    queued = False
    started = time.monotonic()
    last_stats = started
    latest_stats = before
    append_jsonl("run_start", {"family": family, "stats": before})
    try:
        client.queue(family)
        queued = True
        while True:
            now = time.monotonic()
            games = client.pending_games()
            if games and not assigned_ids:
                assigned_ids.update(str(game["game_id"]) for game in games)
                client.leave_queue(family)
                queued = False
                append_jsonl("queue_left_after_assignment", {
                    "family": family, "assigned_ids": sorted(assigned_ids)
                })
            elif games:
                assigned_ids.update(str(game["game_id"]) for game in games)

            for game in games:
                game_id = str(game["game_id"])
                action = strategy_fn(game)
                result = client.move(game_id, action)
                move_results.append({"game_id": game_id, "round": game["game_state"].get("round"),
                                     "action": action, "result": result})
                append_jsonl("move_result", move_results[-1])
                if result.get("valid") is False:
                    raise RuntimeError(f"server rejected move for {game_id}: {result}")
                if result.get("game_over"):
                    completed_ids.add(game_id)

            if not assigned_ids and now - started >= match_timeout:
                append_jsonl("match_timeout", {"family": family, "seconds": match_timeout})
                break

            if now - last_stats >= stats_interval:
                latest_stats = client.stats()
                last_stats = now
                if assigned_ids and int(latest_stats.get("active_games", 0) or 0) == 0:
                    break
            time.sleep(poll_interval)
    finally:
        # Safe even if already removed; essential on every exit path.
        try:
            client.leave_queue(family if queued else None)
        except Exception as exc:
            TELEMETRY_LOG.append({"event": "leave_queue", "error": f"{type(exc).__name__}: {exc}"})
    after = client.stats()
    report = {
        "family": family, "before": before, "after": after,
        "assigned_ids": sorted(assigned_ids),
        "completed_seen_in_move_response": sorted(completed_ids),
        "move_count": len(move_results),
        "assignment_overshoot": max(0, len(assigned_ids) - 1),
    }
    append_jsonl("run_end", report)
    return report


LIVE_BUILD_ID = "v21-controlled-microbatches-2026-08-28"
print("Live build:", LIVE_BUILD_ID)

TARGET_COMPLETIONS = {
    "bargaining": 20,
    "negotiation": 20,
    "persuasion": 20,
}
FAMILY_STOP_LOSS = {
    "bargaining": 12.0,
    "negotiation": 12.0,
    "persuasion": 12.0,
}
MAX_QUEUE_ATTEMPTS_PER_FAMILY = 30
MATCH_TIMEOUT_SECONDS = 300.0
RUN_LIVE = True

def family_score_snapshot(stats, family):
    scores = stats.get("scores") or {}
    item = scores.get(family) or {}
    rating = item.get("rating")
    count = item.get("games_played", item.get("games", item.get("game_count", 0)))
    if rating is None:
        raise KeyError(f"stats missing {family} rating: {stats}")
    return {"rating": finite_float(rating), "games_played": int(count or 0)}

def run_controlled_family_microbatch(client, family, target, stop_loss):
    initial_stats = client.stats()
    initial = family_score_snapshot(initial_stats, family)
    checkpoints = []
    stop_reason = None
    global_abort = False

    for attempt in range(1, MAX_QUEUE_ATTEMPTS_PER_FAMILY + 1):
        current_stats = client.stats()
        current = family_score_snapshot(current_stats, family)
        completed = current["games_played"] - initial["games_played"]
        if completed >= target:
            stop_reason = "target reached"
            break

        fallback_before = len(ACTION_FALLBACK_LOG)
        telemetry_before = len(TELEMETRY_LOG)
        print(
            f"{family}: attempt {attempt}; authoritative completions "
            f"{completed}/{target}; rating change "
            f"{current['rating'] - initial['rating']:+.2f}"
        )
        report = run_one_queue_assignment(
            client, strategy, family, match_timeout=MATCH_TIMEOUT_SECONDS
        )
        after = family_score_snapshot(report["after"], family)
        before = family_score_snapshot(report["before"], family)
        count_delta = after["games_played"] - before["games_played"]
        rating_delta = after["rating"] - before["rating"]
        cumulative_count = after["games_played"] - initial["games_played"]
        cumulative_rating = after["rating"] - initial["rating"]
        new_fallbacks = len(ACTION_FALLBACK_LOG) - fallback_before
        new_telemetry = len(TELEMETRY_LOG) - telemetry_before

        harvested = harvest_completed_games(client)
        credited = credit_rating_delta(
            report["assigned_ids"], family,
            before["rating"], after["rating"], count_delta,
        )
        checkpoint = {
            "family": family, "attempt": attempt,
            "target": target, "count_delta": count_delta,
            "rating_delta": round(rating_delta, 6),
            "cumulative_count": cumulative_count,
            "cumulative_rating": round(cumulative_rating, 6),
            "assignment_overshoot": report["assignment_overshoot"],
            "assigned_ids": report["assigned_ids"],
            "harvested_count": len(harvested),
            "rating_credit": credited,
            "new_action_fallbacks": new_fallbacks,
            "new_telemetry": new_telemetry,
        }
        checkpoints.append(checkpoint)
        append_jsonl("v21_microbatch_checkpoint", checkpoint)
        display(checkpoint)

        if report["assignment_overshoot"] > 0:
            stop_reason = "assignment overshoot"
            global_abort = True
            break
        if new_fallbacks > 0:
            stop_reason = "live action fallback"
            global_abort = True
            break
        if new_telemetry > 0:
            stop_reason = "new telemetry diagnostic"
            global_abort = True
            break
        if count_delta <= 0:
            stop_reason = "no authoritative completion before timeout"
            break
        if cumulative_rating <= -abs(stop_loss):
            stop_reason = "family rating stop-loss"
            break
        if cumulative_count >= target:
            stop_reason = "target reached"
            break
    else:
        stop_reason = "maximum queue attempts reached"

    final_stats = client.stats()
    final = family_score_snapshot(final_stats, family)
    summary = {
        "family": family,
        "initial": initial,
        "final": final,
        "completed": final["games_played"] - initial["games_played"],
        "rating_change": round(final["rating"] - initial["rating"], 6),
        "stop_reason": stop_reason,
        "global_abort": global_abort,
        "checkpoints": checkpoints,
    }
    append_jsonl("v21_family_summary", summary)
    return summary

def run_v21_microbatch_tests():
    class MicroBatchClient:
        def __init__(self, rating_deltas):
            self.rating_deltas = list(rating_deltas)
            self.finished = 0
            self.rating = 1000.0
            self.queued = False
            self.delivered = False

        def stats(self):
            return {
                "active_games": 0,
                "scores": {
                    "bargaining": {
                        "rating": self.rating,
                        "games_played": self.finished,
                    }
                },
            }

        def queue(self, family):
            assert family == "bargaining"
            self.queued = True
            self.delivered = False
            return {"status": "queued"}

        def leave_queue(self, family=None):
            self.queued = False
            return {"status": "left"}

        def pending_games(self):
            if not self.queued or self.delivered:
                return []
            self.delivered = True
            return [base_game("bargaining", "offer", {
                "current_player": "player_1", "money_to_divide": 100.0,
                "delta_1": 0.90, "delta_2": 0.95,
                "complete_information": True, "round": 1,
                "max_rounds": 5, "horizon_known": True,
                "history": [], "messages_allowed": False,
            }, "player_1", f"test-micro-{self.finished + 1}")]

        def move(self, game_id, action):
            validate_action(self.pending_game_template(game_id), action)
            delta = self.rating_deltas[min(self.finished, len(self.rating_deltas) - 1)]
            self.finished += 1
            self.rating += delta
            return {"valid": True, "game_over": True}

        def pending_game_template(self, game_id):
            return base_game("bargaining", "offer", {
                "current_player": "player_1", "money_to_divide": 100.0,
                "delta_1": 0.90, "delta_2": 0.95,
                "complete_information": True, "round": 1,
                "max_rounds": 5, "horizon_known": True,
                "history": [], "messages_allowed": False,
            }, "player_1", game_id)

        def game_state(self, game_id):
            return {"player_1_payoff": 50.0}

    clean = MicroBatchClient([1.0, 1.5, -0.5])
    summary = run_controlled_family_microbatch(clean, "bargaining", 3, 12.0)
    assert summary["completed"] == 3
    assert summary["stop_reason"] == "target reached"
    assert not summary["global_abort"]

    losing = MicroBatchClient([-13.0, 5.0])
    summary = run_controlled_family_microbatch(losing, "bargaining", 5, 12.0)
    assert summary["completed"] == 1
    assert summary["stop_reason"] == "family rating stop-loss"
    assert not summary["global_abort"]
    print("V21 repeated micro-batch target and stop-loss tests passed.")

if not RUN_LIVE:
    run_v21_microbatch_tests()

if RUN_LIVE:
    api_key = configure_glee_api_key()
    client = GleeClient(api_key=api_key)
    v21_family_summaries = []
    for family, target in TARGET_COMPLETIONS.items():
        print(f"Starting controlled {family} micro-batch: target={target}")
        summary = run_controlled_family_microbatch(
            client, family, target, FAMILY_STOP_LOSS[family]
        )
        v21_family_summaries.append(summary)
        display(summary)
        if summary["global_abort"]:
            print("Aborting remaining families:", summary["stop_reason"])
            break
    report_paths = export_session_report(
        "glee_v21_session_report.json", "glee_v21_rating_outcomes.csv"
    )
    print("Family summaries completed:", len(v21_family_summaries))
    print("Session reports:", report_paths)
else:
    print("Live matchmaking is disabled by RUN_LIVE=False.")


## V13 integration test — controlled queue exposure

This test does not contact the competition. It verifies the low-level runner using a local mock client.


In [ ]:
def run_v13_runner_test():
    class OneAssignmentClient:
        def __init__(self, pending_game):
            self.pending_game = pending_game
            self.pending_calls = 0
            self.queue_calls = []
            self.leave_calls = []
            self.moves = []
            self.finished = False

        def stats(self):
            return {"active_games": 0,
                    "scores": {"bargaining": {"rating": 1000,
                                                "games_played": int(self.finished)}}}

        def queue(self, family):
            self.queue_calls.append(family)
            return {"status": "queued", "game_family": family}

        def leave_queue(self, family=None):
            self.leave_calls.append(family)
            return {"status": "left"}

        def pending_games(self):
            self.pending_calls += 1
            return [self.pending_game] if self.pending_calls == 1 else []

        def move(self, game_id, action):
            self.moves.append((game_id, action))
            self.finished = True
            return {"valid": True, "game_over": True,
                    "result": {"outcome": "agreement"}}

    template = base_game(
        "bargaining", "offer",
        {"phase": "offer", "current_player": "player_1", "proposer": "player_1",
         "round": 1, "max_rounds": 5, "horizon_known": True,
         "money_to_divide": 100, "delta_1": 0.95, "delta_2": 0.95,
         "last_offer": None, "messages_allowed": False,
         "complete_information": True},
        game_id="v13-runner-mock",
    )
    mock = OneAssignmentClient(template)
    report = run_one_queue_assignment(
        mock, strategy, "bargaining", poll_interval=0,
        match_timeout=1, stats_interval=0
    )
    assert mock.queue_calls == ["bargaining"]
    assert mock.leave_calls and mock.leave_calls[0] == "bargaining"
    assert len(mock.moves) == 1
    assert report["assigned_ids"] == ["v13-runner-mock"]
    assert report["assignment_overshoot"] == 0
    print("V13 one-queue integration test passed.")

run_v13_runner_test()


## Inspect exact evidence


In [ ]:
print("Run mode:", RUN_MODE)
print("Live action fallback count:", len(ACTION_FALLBACK_LOG))
display(list(ACTION_FALLBACK_LOG))

print("Telemetry diagnostic count:", len(TELEMETRY_LOG))
display(list(TELEMETRY_LOG)[-30:])

print("Completed-game policy evidence:")
display(policy_evidence_rows())

print("Recent non-synthetic assignments:")
live_assignments = [(game_id, data) for game_id, data in POLICY_ASSIGNMENTS.items()
                    if not data.get("synthetic")]
display(live_assignments[-50:])

print("Recent decisions:")
display(list(DECISION_LOG)[-30:])
print("Bargaining profiles:", len(BARGAINING_MEMORY))
print("Negotiation profiles:", len(NEGOTIATION_MEMORY))
print("Role-separated persuasion profiles:", len(PERSUASION_MEMORY))
print("Persistent state:", POLICY_STATE_PATH.resolve())
print("Append-only evidence:", EVIDENCE_PATH.resolve())
